In [ ]:
def plot_hi_het_triangle(
    all_generations_data: Dict[str, List[Dict[str, Any]]],
    plot_mode: Literal['individuals', 'highlight_selected', 'means'] = 'individuals',
    highlight_gen: Optional[str] = None,
    show_individual_points_behind_means: bool = False,
    save_filename: Optional[str] = None
):
    """
    Plots Hybrid Index (HI) versus Heterozygosity (HET) for simulated generations,
    offering various display options for detailed analysis.

    Args:
        all_generations_data (dict): A dictionary containing all simulation data.
                                     Keys are generation names (e.g., 'Ancestry_1', 'F2', 'BC1A'),
                                     and values are lists of dictionaries, each containing
                                     'HI' and 'HET' values for individual organisms within that generation.
        plot_mode (Literal): Determines what to plot:
            - 'individuals': Plots all individual points for every generation with distinct colors.
            - 'highlight_selected': Plots individual points, highlighting one specific generation
                                    in color while others appear in grayscale. Requires `highlight_gen`.
            - 'means': Displays only the mean HI and HET values for each generation, with labels.
                       Can optionally show background individual points via `show_individual_points_behind_means`.
        highlight_gen (str, optional): The name of the generation to highlight if `plot_mode` is
                                       'highlight_selected' (e.g., 'F4', 'BC1A'). Ignored otherwise.
        show_individual_points_behind_means (bool): If `plot_mode` is 'means', setting this to True
                                                    will plot all individual points in a very light
                                                    grayscale behind the mean points for context. Defaults to False.
        save_filename (str, optional): If provided, the generated plot will be saved to this filename.
                                       Otherwise, it will be displayed.
    """
    # Set up the plot figure and axes with a chosen size for clarity.
    fig, ax = plt.subplots(figsize=(10, 8))

    # Remove the top and right spines to give the plot a cleaner aesthetic.
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Make the remaining spines slightly thicker for more prominence.
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)

    # Label the axes clearly for easy understanding of the plot content.
    ax.set_xlabel("Hybrid Index (proportion of the selected allele)", fontsize=12)
    ax.set_ylabel("Heterozygosity (proportion heterozygous loci)", fontsize=12)

    # --- Common Helper Functions and Data Structures ---

    def sort_key(label: str):
        """
        Custom sorting key for generation labels to ensure a logical order in plots and legends.
        Assigns numerical priorities to specific generation types (Ancestry, F1, F-series, BC-series).
        """
        if label == 'Ancestry_1': return (0, label) # Ancestry_1 comes first
        if label == 'Ancestry_2': return (1, label) # Ancestry_2 comes second
        if label == 'F1': return (2, label) # F1 comes third
        
        # For F generations (e.g., F2, F3), extract the number for numerical sorting.
        match_f = re.match(r'F(\d+)', label)
        if match_f:
            return (3, int(match_f.group(1)))
        # Fallback for any F labels not matching the standard pattern, placing them at the end of F-group.
        elif label.startswith('F'):
            return (3, float('inf'), label) 
        
        # For BC (Backcross) generations (e.g., BC1A, BC2B), extract number and handle suffix.
        match_bc = re.match(r'BC(\d+)(_?[AB]?)', label) # Adjusted regex for optional underscore
        if match_bc:
            num_part = int(match_bc.group(1))
            suffix_part = match_bc.group(2) if match_bc.group(2) else ''
            # Prioritize 'A' suffix over 'B' if both exist for the same number
            if 'A' in suffix_part:
                return (4, num_part, 'A')
            elif 'B' in suffix_part:
                return (4, num_part, 'B')
            else:
                return (4, num_part, '') # For BCs without A/B suffix
        # Fallback for any BC labels not matching the standard pattern, placing them at the end of BC-group.
        elif label.startswith('BC'):
            return (4, float('inf'), label)
        
        # Any other unknown labels will go at the very end.
        return (5, label)

    # Get all generation names from the data and sort them using the custom key.
    sorted_gen_names = sorted(list(all_generations_data.keys()), key=sort_key)

    # Set up a cycle of default colors to ensure a good variety for generations.
    default_colors_cycle = itertools.cycle([
        'blue', 'red', 'green', 'purple', 'orange', 'brown', 'pink', 'teal',
        'darkviolet', 'darkcyan', 'lime', 'gold', 'navy', 'maroon',
        'darkgreen', 'darkred', 'darkblue', 'darkgoldenrod', 'darkslategray',
        'cornflowerblue', 'olivedrab', 'peru', 'rosybrown', 'salmon',
        'seagreen', 'sienna', 'darkkhaki', 'mediumorchid', 'lightcoral'
    ])
    # This dictionary will store the assigned color for each generation for consistency.
    color_map = {}

    def get_color_for_mode(gen_name):
        """
        Determines the plotting color for a given generation. Prioritizes specific colors
        for Ancestry_1, Ancestry_2, and F1, then uses the default color cycle.
        """
        if gen_name not in color_map:
            if gen_name == 'Ancestry_1': color_map[gen_name] = 'black'
            elif gen_name == 'Ancestry_2': color_map[gen_name] = 'grey'
            elif gen_name == 'F1': color_map[gen_name] = 'purple'
            else: color_map[gen_name] = next(default_colors_cycle)
        return color_map[gen_name]

    # Initialize an empty list to hold the elements that will appear in the plot legend.
    legend_elements = []

    # --- Conditional Plotting Logic based on `plot_mode` ---

    if plot_mode == 'individuals':
        # In 'individuals' mode, plot every single individual point for all generations.
        
        # Loop through each sorted generation to extract and plot its individual data.
        for gen_name in sorted_gen_names:
            values = all_generations_data.get(gen_name, [])
            if not values: # Skip if no data is available for this generation.
                print(f"Skipping plotting for {gen_name} as no data found.")
                continue

            # Extract Hybrid Index and Heterozygosity values, ensuring they are not None.
            hi_values = [d['HI'] for d in values if 'HI' in d and d['HI'] is not None]
            het_values = [d['HET'] for d in values if 'HET' in d and d['HET'] is not None]

            if hi_values and het_values: # Only plot if valid data for both HI and HET exists.
                # Get the appropriate color for this generation.
                color = get_color_for_mode(gen_name)
                # Plot the individual points.
                ax.scatter(hi_values, het_values,
                           color=color,
                           alpha=0.7, # Transparency helps visualize dense areas.
                           marker='o', # Use circles for points.
                           s=20, # Standard size for individual points.
                           zorder=2) # Ensure points are above the triangle lines.
                
                # Add an entry for this generation to the legend.
                legend_elements.append(plt.Line2D([0], [0], marker='o', color='w',
                                                 markerfacecolor=color, markersize=8,
                                                 alpha=0.7, label=gen_name))
            else:
                print(f"Skipping plotting for {gen_name} as no valid HI/Het data was found.")

    elif plot_mode == 'highlight_selected':
        # In 'highlight_selected' mode, highlight one specific generation and grayscale others.
        if highlight_gen is None:
            print("Error: When 'highlight_selected' mode is chosen, 'highlight_gen' must be specified.")
            plt.close(fig) # Close the figure as plotting cannot proceed meaningfully.
            return

        # Define specific styling parameters for highlighted and grayscale points.
        HIGHLIGHT_COLOR = 'dodgerblue'
        HIGHLIGHT_ALPHA = 1.0
        HIGHLIGHT_MARKER_SIZE = 35

        GRAYSCALE_COLOR = 'silver'
        GRAYSCALE_ALPHA = 0.2
        GRAYSCALE_MARKER_SIZE = 20

        # Loop through all generations to apply the highlighting logic during plotting.
        for gen_name in sorted_gen_names:
            values = all_generations_data.get(gen_name, [])
            if not values: continue # Skip if no data.

            hi_values = [d['HI'] for d in values if 'HI' in d and d['HI'] is not None]
            het_values = [d['HET'] for d in values if 'HET' in d and d['HET'] is not None]

            if hi_values and het_values: # Only proceed if valid data exists.
                # Determine the style (color, alpha, size, zorder) based on whether
                # the current generation is the one to be highlighted.
                if gen_name == highlight_gen:
                    color = HIGHLIGHT_COLOR
                    alpha = HIGHLIGHT_ALPHA
                    s = HIGHLIGHT_MARKER_SIZE
                    zorder = 5 # Highlighted generation should be on top.
                else:
                    color = GRAYSCALE_COLOR
                    alpha = GRAYSCALE_ALPHA
                    s = GRAYSCALE_MARKER_SIZE
                    zorder = 2 # Other generations are in the background.

                # Plot the individual points with the determined style.
                ax.scatter(hi_values, het_values,
                           color=color,
                           alpha=alpha,
                           marker='o',
                           s=s,
                           zorder=zorder)
                
                # Add entries to the legend, ensuring no duplicates for generations that might
                # be plotted in both highlighted and grayscale states (though typically only one highlight).
                if gen_name not in [el.get_label() for el in legend_elements]:
                    legend_elements.append(plt.Line2D([0], [0], marker='o', color='w',
                                                     markerfacecolor=color, markersize=8,
                                                     alpha=alpha, label=gen_name))
            else:
                print(f"Skipping plotting for {gen_name} as no valid HI/Het data was found.")

    elif plot_mode == 'means':
        # In 'means' mode, plot only the mean HI and HET for each generation.
        # Adjust axis labels for clarity when plotting means.
        ax.set_xlabel("Mean Hybrid Index (proportion A alleles)", fontsize=12)
        ax.set_ylabel("Mean Heterozygosity (proportion heterozygous loci)", fontsize=12)

        if show_individual_points_behind_means:
            # If requested, plot all individual points as subtle background context.
            for gen_name in sorted_gen_names:
                values = all_generations_data.get(gen_name, [])
                if not values: continue # Skip if no data.

                hi_values = [d['HI'] for d in values if 'HI' in d and d['HI'] is not None]
                het_values = [d['HET'] for d in values if 'HET'] and d['HET'] is not None]
                
                if hi_values and het_values:
                    # Use a very light grey and low alpha for subtle background points.
                    ax.scatter(hi_values, het_values, color='lightgray', alpha=0.1, s=10, zorder=1)
                else:
                    print(f"Skipping individual background points for {gen_name} due to missing data.")

        # Now, loop through each generation to calculate and plot its mean.
        for gen_name in sorted_gen_names:
            values = all_generations_data.get(gen_name, [])
            if not values: # Report and skip if a generation has no data for mean calculation.
                print(f"Skipping mean plot for {gen_name} due to missing data.")
                continue

            hi_values = [d['HI'] for d in values if 'HI' in d and d['HI'] is not None]
            het_values = [d['HET'] for d in values if 'HET' in d and d['HET'] is not None]

            # Only calculate and plot means if valid HI and HET data are available.
            if hi_values and het_values:
                mean_hi = np.mean(hi_values)   # Calculate the mean Hybrid Index.
                mean_het = np.mean(het_values) # Calculate the mean Heterozygosity.
                
                color = get_color_for_mode(gen_name) # Get the consistent color for this generation.
                
                # Set marker size; anchor points (Ancestry, F1) are slightly larger.
                marker_size = 80
                if gen_name in ['Ancestry_1', 'Ancestry_2', 'F1']:
                    marker_size = 100

                # Plot the mean point with a black edge for better definition.
                ax.scatter(mean_hi, mean_het, color=color, s=marker_size, marker='o', edgecolors='black', linewidth=1.5, zorder=3)
                
                # Add a text label next to the mean point for easy identification.
                ax.text(mean_hi + 0.01, mean_het + 0.01, gen_name, fontsize=9, color=color, ha='left', va='bottom', zorder=4)
                
                # Add this generation to the legend, avoiding duplicates.
                if gen_name not in [el.get_label() for el in legend_elements]:
                    legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=8, label=gen_name))
            else:
                print(f"Skipping mean plot for {gen_name} as no valid HI/HET data was found to calculate mean.")
    else:
        # If an invalid plot mode is provided, inform the user and close the figure.
        print(f"Error: Invalid plot_mode '{plot_mode}'. It must be 'individuals', 'highlight_selected', or 'means'.")
        plt.close(fig)
        return

    # --- Common Plot Elements (Applied regardless of plot_mode) ---
    
    # Define and draw the theoretical triangle edges on the plot. These represent the boundaries
    # of the possible HI/HET space for a two-ancestor cross, with pure ancestors at corners
    # (0,0) and (1,0) and the F1 population at the apex (0.5, 1.0).
    triangle_edges = [
        [(0.0, 0.0), (0.5, 1.0)], # Edge from Ancestry_1 corner to F1 apex
        [(0.5, 1.0), (1.0, 0.0)], # Edge from F1 apex to Ancestry_2 corner
        [(0.0, 0.0), (1.0, 0.0)]  # Base edge between Ancestry_1 and Ancestry_2
    ]
    for (x0, y0), (x1, y1) in triangle_edges:
        # Plot each edge as a grey line, positioned at the very back (lowest zorder).
        ax.plot([x0, x1], [y0, y1], linestyle='-', color='gray', linewidth=1.5, alpha=0.7, zorder=0)

    # Set the limits for X and Y axes to slightly extend beyond 0 and 1,
    # ensuring all points and labels are fully visible and not cut off.
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    # Ensure the aspect ratio is equal so the triangle shape is not distorted.
    ax.set_aspect('equal', adjustable='box')

    # Adjust the subplot layout to make room for the legend on the right side.
    plt.subplots_adjust(left=0.1, right=0.75, top=0.9, bottom=0.1)

    # Add the legend to the plot, positioning it outside the main plot area
    # for better clarity and ensuring it does not have a visible frame.
    ax.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.35, 1), fontsize=10, frameon=False)

    # If a filename is provided, save the plot before displaying it.
    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight') # 'tight' ensures all plot elements fit.
        print(f"Plot saved to {save_filename}")

    # Finally, display the generated plot.
    plt.show()